In [ ]:
#jembatan

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Parameter
max_words = 10000  # Gunakan 10rb kata paling populer
max_len = 100      # Maksimal 100 kata per kalimat

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df_final['text']) # nama kolom teks

#pastikan menggunakan df_final, bukan df karena df_final sudah melalui proses pembersihan dan penggabungan data
sequences = tokenizer.texts_to_sequences(df_final['text']) 
X = pad_sequences(sequences, maxlen=max_len, padding='post')
y = df_final['emotion_id'].values

In [ ]:
# membangun arsitektur model

embedding_dim = 128
num_classes = len(np.unique(y))

model = tf.keras.Sequential([
    # Layer Input/Embedding
    Embedding(max_words, embedding_dim, input_length=max_len),
    
    # Layer BiLSTM (Penting: return_sequences=True agar Attention bisa bekerja)
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.3),
    
    # Custom Attention Layer Anda
    AttentionLayer(),
    
    # Dense Layer & Output
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

In [ ]:
import tensorflow as tf

# Persiapan data training dan validasi
X_train_raw = df_final_train['text_akhir']
y_train = df_final_train['emotion_id'].values
sequences_train = tokenizer.texts_to_sequences(X_train_raw)
X_train = pad_sequences(sequences_train, maxlen=max_len, padding='post')

X_val_raw = df_final_val['text_akhir']
y_val = df_final_val['emotion_id'].values
sequences_val = tokenizer.texts_to_sequences(X_val_raw)
X_val = pad_sequences(sequences_val, maxlen=max_len, padding='post')

# Inisialisasi Optimizer & Loss
initial_lr = 1e-3
opt = tf.keras.optimizers.Adam(learning_rate=initial_lr)
loss_fn = WeightedFocalLoss(class_weights=class_weights_dict)

# Konversi ke Dataset TF
train_dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train)).shuffle(1024).batch(32)
val_dataset = tf.data.Dataset.from_tensor_slices((X_val, y_val)).batch(32)

# Metrik
train_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()
val_acc_metric = tf.keras.metrics.SparseCategoricalAccuracy()

# Parameter Penurun Learning Rate (Manual ReduceLROnPlateau)
wait = 0
patience = 3
best_val_acc = 0.0
lr_factor = 0.5  # Faktor pengali (turun 50%)

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss_value = loss_fn(y, logits)
    grads = tape.gradient(loss_value, model.trainable_weights)
    opt.apply_gradients(zip(grads, model.trainable_weights))
    train_acc_metric.update_state(y, logits)
    return loss_value

# Eksekusi Loop Pelatihan Manual
for epoch in range(20):
    print(f"\nMemulai Epoch {epoch + 1}")
    
    # Training Loop
    for x_batch, y_batch in train_dataset:
        loss = train_step(x_batch, y_batch)
    
    # Validation Loop (Manual)
    for x_val, y_val in val_dataset:
        val_logits = model(x_val, training=False)
        val_acc_metric.update_state(y_val, val_logits)
    
    # Ambil hasil metrik
    train_acc = train_acc_metric.result()
    val_acc = val_acc_metric.result()
    current_lr = opt.learning_rate.numpy()

    print(f"Loss: {loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f} | LR: {current_lr:.6f}")

    # Logika Penurunan Learning Rate (Reduce LR on Plateau)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        wait = 0  # Reset hitungan jika ada peningkatan
    else:
        wait += 1 # Tambah hitungan jika stagnan
        if wait >= patience:
            new_lr = current_lr * lr_factor
            opt.learning_rate.assign(new_lr)
            print(f">>> Akurasi stagnan selama {patience} epoch. Menurunkan LR ke: {new_lr:.6f}")
            wait = 0

    # Cek Target Akurasi Utama
    if train_acc >= 0.85:
        print(f"Target 85% tercapai pada Epoch {epoch+1}!")
        break
    
    # Reset metrik untuk epoch berikutnya
    train_acc_metric.reset_state()
    val_acc_metric.reset_state()